# Scraper USP Primo – **Versão final (com parada automática)**
Agora lê o número total de resultados (ex.: 9.724) e encerra automaticamente.
Inclui todas as funções de enriquecimento.

In [1]:
# ==================== CONFIGURAÇÃO ====================
import os, re, json, time, random, logging
from datetime import datetime
from urllib.parse import urljoin, urlparse, parse_qs, urlencode, urlunparse

import requests
import pandas as pd
from bs4 import BeautifulSoup

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

URL_BUSCA = (
    "https://buscaintegrada.usp.br/primo_library/libweb/action/search.do"
    "?fn=search&ct=search&initialSearch=true&mode=Advanced"
    "&tab=default_tab&indx=1&dum=true&srt=rank&vid=USP&tb=t"
    "&vl(4708870UI0)=sub&vl(1UIStartWith0)=sub&vl(1UIStartWith1)=contains"
    "&vl(freeText1)=sociologia%20das%20elites"
    "&vl(4708301UI6)=01&vl(4708302UI6)=01"
    "&vl(4708303UI6)=1970&vl(4708304UI6)=31"
    "&vl(4708305UI6)=12&vl(4708306UI6)=9999&Submit=Buscar"
)

PAGE_SIZE = 50           # tamanho de página desejado (o servidor pode ignorar)
THROTTLE = 1.5           # segundos entre requisições
MAX_PAGES = None         # limite de páginas (None = até o total)
MAX_ENRIQUECER = 20      # quantos registros enriquecer (None = todos)

CHECKPOINT_DIR = "checkpoints"
OUTPUT_DIR = "saida"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.0 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
]
session = requests.Session()
def nova_sessao():
    session.headers.update({
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    })
nova_sessao()

In [2]:
def http_retry(url, timeout=30, max_tentativas=5):
    for tent in range(1, max_tentativas+1):
        try:
            nova_sessao()
            r = session.get(url, timeout=timeout)
            if r.status_code == 200:
                return r
            if r.status_code == 429:
                delay = 15 * tent
                log.warning(f"429 – aguardando {delay}s")
                time.sleep(delay)
            else:
                log.warning(f"Status {r.status_code}, tentativa {tent}")
        except Exception as e:
            log.error(f"Erro: {e}")
        time.sleep(2 ** tent)
    return None

In [3]:
def extrair_itens_html(html):
    soup = BeautifulSoup(html, "html.parser")
    itens, seen = [], set()
    seletores = ["a.EXLResultTitle", "a.EXLBriefResultsDisplayTitle", "a[title='Detalhes']",
                 "h2.EXLResultTitle a", "h3 a[href*='display.do']", "a[href*='fulldisplay']",
                 "a[class*='recordTitle']", "a[data-recordid]", "a[aria-label*='Detalhes']",
                 "a[aria-label*='Details']", "div.EXLResultTitle a"]
    links = []
    for sel in seletores:
        links.extend(soup.select(sel))
    if not links:
        for a in soup.select("a[href]"):
            if any(k in a.get("href","") for k in ("display.do", "fulldisplay")) and len(a.get_text(strip=True))>3:
                links.append(a)
    for a in links:
        href = a.get("href","")
        title = a.get_text(strip=True)
        if not href or not title:
            continue
        url_reg = urljoin(URL_BUSCA, href)
        if url_reg in seen:
            continue
        seen.add(url_reg)
        low = title.lower()
        if any(w in low for w in ["próximo","anterior","detalhes","refinar","acesso","login","entrar"]):
            continue
        itens.append({"title": title, "record_url": url_reg})
    return itens

def extrair_total_resultados(html):
    """Extrai o número total de resultados da página (ex.: 9.724)."""
    soup = BeautifulSoup(html, "html.parser")
    texto = soup.get_text(" ", strip=True)
    m = re.search(r"de (\d{1,3}(?:\.\d{3})*|\d+)", texto)
    if m:
        return int(m.group(1).replace(".", ""))
    return None

def url_com_indx(url, indx):
    pu = urlparse(url)
    qs = parse_qs(pu.query, keep_blank_values=True)
    qs["indx"] = [str(max(1, indx))]
    return urlunparse((pu.scheme, pu.netloc, pu.path, pu.params, urlencode(qs, doseq=True), pu.fragment))

def salvar_checkpoint(obj, nome):
    with open(os.path.join(CHECKPOINT_DIR, f"{nome}.json"), "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def coletar_via_html(url_busca, page_size=50, throttle=1.5, max_pages=None):
    # Monta URL limpa com page_size
    pu = urlparse(url_busca)
    qs = parse_qs(pu.query, keep_blank_values=True)
    qs["indx"] = ["1"]
    qs["vl(47417594UI0)"] = [str(page_size)]
    for p in ["initialSearch", "dum", "Submit"]:
        qs.pop(p, None)
    base_url = urlunparse((pu.scheme, pu.netloc, pu.path, pu.params, urlencode(qs, doseq=True), pu.fragment))

    itens = []
    total_geral = None
    pagina = 0
    indx = 1
    falhas = 0
    while True:
        if max_pages and pagina >= max_pages:
            break
        log.info(f"Página {pagina+1} (indx={indx})")
        resp = http_retry(url_com_indx(base_url, indx))
        if not resp:
            falhas += 1
            if falhas >= 3:
                log.error("Muitas falhas. Parando.")
                break
            indx += max(10, page_size)
            continue
        falhas = 0
        page_items = extrair_itens_html(resp.text)
        if total_geral is None:
            total_geral = extrair_total_resultados(resp.text)
            log.info(f"Total de resultados informado: {total_geral}")
        if not page_items:
            log.info("Nenhum item. Fim.")
            break
        itens.extend(page_items)
        pagina += 1
        log.info(f"  {len(page_items)} itens – acumulado {len(itens)} de {total_geral}")
        salvar_checkpoint(itens, f"html_pag_{pagina}")
        # Parar se já tivermos o total de itens
        if total_geral and len(itens) >= total_geral:
            log.info("Atingido o total de resultados. Parando.")
            break
        # Próximo indx
        step = len(page_items) if len(page_items) >= 10 else 10
        indx += step
        time.sleep(throttle)
    return itens

In [4]:
log.info("Coletando via HTML (com parada automática)...")
itens = coletar_via_html(URL_BUSCA, page_size=PAGE_SIZE, throttle=THROTTLE, max_pages=MAX_PAGES)
log.info(f"Total coletado: {len(itens)}")

with open(os.path.join(OUTPUT_DIR, f"itens_brutos_{datetime.now():%Y-%m-%d}.json"), "w", encoding="utf-8") as f:
    json.dump(itens, f, ensure_ascii=False, indent=2)
log.info("Itens brutos salvos.")

2026-05-09 15:43:27,553 [INFO] Coletando via HTML (com parada automática)...
2026-05-09 15:43:27,553 [INFO] Página 1 (indx=1)
2026-05-09 15:43:28,342 [INFO] Total de resultados informado: 1
2026-05-09 15:43:28,342 [INFO]   9 itens – acumulado 9 de 1
2026-05-09 15:43:28,343 [INFO] Atingido o total de resultados. Parando.
2026-05-09 15:43:28,343 [INFO] Total coletado: 9
2026-05-09 15:43:28,345 [INFO] Itens brutos salvos.


In [5]:
# ==================== FUNÇÕES DE ENRIQUECIMENTO (COMPLETAS) ====================
try:
    from dateutil import parser as dateparser
except ImportError:
    dateparser = None

KEYS_LISTLIKE = {"author", "citation_author", "dc.creator",
                 "keywords", "citation_keywords", "dc.subject"}
KEYS_DATE = {"date", "citation_date", "citation_publication_date",
             "dc.date", "dc.date.issued", "dc.date.created", "prism.publicationdate"}

def _norm_key(k):
    k = (k or "").strip().lower()
    k = k.replace("http-equiv:", "").replace("property:", "").replace("name:", "").strip()
    k = k.replace(":", ".")
    k = re.sub(r"[^a-z0-9.]+", "_", k).strip("_")
    return k

def _split_keywords(val):
    parts = re.split(r"[;,/•|]\s*|\s{2,}", val or "")
    parts = [re.sub(r"^(palavras?-chave|keywords)\s*:\s*", "", p, flags=re.I).strip(" .,;-") 
             for p in parts if p and p.strip()]
    seen, out = set(), []
    for p in parts:
        pl = p.lower()
        if pl not in seen:
            out.append(p); seen.add(pl)
    return out

def _parse_date(val):
    raw = (val or "").strip()
    if not raw:
        return {"date_raw": None, "date_iso": None, "year": None}
    if dateparser:
        try:
            dt = dateparser.parse(raw, fuzzy=True)
            return {"date_raw": raw, "date_iso": dt.date().isoformat(), "year": str(dt.year)}
        except Exception:
            pass
    m = re.search(r"\b(19|20)\d{2}-\d{2}-\d{2}\b", raw)
    if m:
        try:
            d = datetime.strptime(m.group(0), "%Y-%m-%d").date()
            return {"date_raw": raw, "date_iso": d.isoformat(), "year": str(d.year)}
        except Exception:
            pass
    m = re.search(r"\b(19|20)\d{2}\b", raw)
    return {"date_raw": raw, "date_iso": None, "year": m.group(0) if m else None}

def extract_and_normalize_meta(html):
    soup = BeautifulSoup(html, "html.parser")
    meta_out = {}
    tags = soup.find_all("meta")
    for t in tags:
        key = t.get("name") or t.get("property") or t.get("http-equiv")
        val = t.get("content") or t.get("value")
        if not key or not val:
            continue
        k = _norm_key(key)
        if not k:
            continue
        is_keywords = (k in {"keywords", "citation_keywords", "dc.subject"})
        is_date = (k in KEYS_DATE)
        is_listlike = (k in KEYS_LISTLIKE)
        if is_keywords:
            vals = _split_keywords(val)
            if vals:
                meta_out.setdefault("keywords", [])
                for v in vals:
                    if v not in meta_out["keywords"]:
                        meta_out["keywords"].append(v)
            continue
        if is_date:
            parsed = _parse_date(val)
            meta_out.setdefault("date_raw", parsed["date_raw"])
            meta_out.setdefault("date_iso", parsed["date_iso"])
            meta_out.setdefault("year", parsed["year"])
            if parsed["date_iso"] and not meta_out.get("date_iso"):
                meta_out["date_iso"] = parsed["date_iso"]
            if parsed["year"] and not meta_out.get("year"):
                meta_out["year"] = parsed["year"]
            continue
        if is_listlike:
            meta_out.setdefault(k, [])
            v = val.strip()
            if v and v not in meta_out[k]:
                meta_out[k].append(v)
        else:
            meta_out.setdefault(k, val.strip())
    # Normaliza aliases
    if "author" in meta_out and isinstance(meta_out["author"], list):
        meta_out["authors"] = meta_out["author"]
    elif "citation_author" in meta_out:
        meta_out["authors"] = meta_out["citation_author"]
    if "keywords" in meta_out and isinstance(meta_out["keywords"], list):
        meta_out["keywords_str"] = "; ".join(meta_out["keywords"])
    if "citation_pdf_url" in meta_out:
        meta_out["pdf_url"] = meta_out["citation_pdf_url"]
    if "citation_abstract" in meta_out and "abstract" not in meta_out:
        meta_out["abstract"] = meta_out["citation_abstract"]
    if "citation_title" in meta_out and "title" not in meta_out:
        meta_out["title"] = meta_out["citation_title"]
    return meta_out

def extrair_campos_visiveis(html):
    soup = BeautifulSoup(html, "html.parser")
    campos = {}
    rotulo_map = {
        "tipo de documento": "tipo_documento",
        "document type": "tipo_documento",
        "instituição": "instituicao",
        "institution": "instituicao",
        "universidade": "instituicao",
        "university": "instituicao",
        "metodologia": "metodologia",
        "methodology": "metodologia",
        "orientador": "orientador",
        "advisor": "orientador",
        "orientadora": "orientador",
        "programa": "programa",
        "program": "programa",
        "área de concentração": "area_concentracao",
        "area of concentration": "area_concentracao",
        "área de conhecimento": "area_concentracao",
        "campo disciplinar": "area_concentracao",
        "assunto": "assunto",
        "subject": "assunto",
        "palavras-chave": "palavras_chave_visivel",
        "keywords": "palavras_chave_visivel",
        "resumo": "resumo",
        "abstract": "resumo",
        "descrição": "descricao",
        "description": "descricao",
    }
    detail_sections = soup.select("div.EXLDetailsContainer, div.EXLResultDetails, dl.EXLDetailsList")
    if not detail_sections:
        detail_sections = soup.find_all(["dl", "div"])
    for section in detail_sections:
        dts = section.find_all(["dt", "span", "div"], class_=re.compile(r"EXLDetailProperty|detail_label", re.I))
        dds = section.find_all(["dd", "span", "div"], class_=re.compile(r"EXLDetailValue|detail_value", re.I))
        if not dts:
            dts = section.find_all("dt")
        if not dds:
            dds = section.find_all("dd")
        for dt, dd in zip(dts, dds):
            rotulo = dt.get_text(strip=True).lower().strip(": ").strip()
            valor = dd.get_text(strip=True)
            chave = rotulo_map.get(rotulo)
            if chave:
                if chave in campos:
                    campos[chave] += "; " + valor
                else:
                    campos[chave] = valor
    return campos

def mesclar_metadados(meta_tags: dict, visiveis: dict) -> dict:
    merged = dict(meta_tags)
    for key, val in visiveis.items():
        if val:
            merged[key] = val
    tipo = merged.get("tipo_documento", "")
    hierarquia = None
    tipo_lower = tipo.lower()
    if "mestrado" in tipo_lower:
        hierarquia = "Mestrado"
    elif "doutorado" in tipo_lower or "phd" in tipo_lower:
        hierarquia = "Doutorado"
    elif "tese" in tipo_lower:
        hierarquia = "Tese"
    elif "dissertação" in tipo_lower:
        hierarquia = "Mestrado"
    elif "artigo" in tipo_lower or "article" in tipo_lower:
        hierarquia = "Artigo"
    elif "livro" in tipo_lower or "book" in tipo_lower:
        hierarquia = "Livro"
    if hierarquia:
        merged["hierarquia"] = hierarquia
    return merged

def row_from_meta(meta: dict, fallback: dict = None) -> dict:
    fb = fallback or {}
    authors = meta.get("authors")
    if isinstance(authors, list):
        authors_str = "; ".join(a.strip() for a in authors if a and str(a).strip())
    else:
        authors_str = authors or ""
    keywords = meta.get("keywords")
    if isinstance(keywords, list):
        keywords_str = "; ".join(k.strip() for k in keywords if k and str(k).strip())
    else:
        keywords_str = keywords or meta.get("keywords_str") or meta.get("palavras_chave_visivel") or ""
    author_list = meta.get("authors")
    if isinstance(author_list, list):
        author_count = len(author_list)
    elif authors_str:
        author_count = len([a for a in authors_str.split(";") if a.strip()])
    else:
        author_count = 0
    return {
        "title": meta.get("title") or fb.get("title", ""),
        "record_url": fb.get("record_url") or meta.get("record_url", ""),
        "authors": authors_str,
        "author_count": author_count,
        "year": meta.get("year") or "",
        "date_iso": meta.get("date_iso") or "",
        "palavras_chave": keywords_str,
        "abstract": meta.get("abstract") or meta.get("resumo") or "",
        "pdf_url": meta.get("pdf_url") or meta.get("citation_pdf_url") or "",
        "metodologia": meta.get("metodologia") or "",
        "tipo_documento": meta.get("tipo_documento") or "",
        "instituicao": meta.get("instituicao") or "",
        "hierarquia": meta.get("hierarquia") or "",
        "area_concentracao": meta.get("area_concentracao") or "",
        "full_data": meta
    }

In [6]:
if MAX_ENRIQUECER:
    log.info(f"Enriquecendo até {MAX_ENRIQUECER} registros...")
    rows = []
    erros = []
    limite = min(len(itens), MAX_ENRIQUECER)
    for i in range(limite):
        item = itens[i]
        log.info(f"{i+1}/{limite}: {item['title'][:60]}")
        url = item["record_url"]
        if not url:
            rows.append(row_from_meta({}, fallback=item))
            continue
        resp = http_retry(url, timeout=20, max_tentativas=3)
        if not resp:
            erros.append(url)
            rows.append(row_from_meta({}, fallback=item))
        else:
            meta_tags = extract_and_normalize_meta(resp.text)
            visiveis = extrair_campos_visiveis(resp.text)
            merged = mesclar_metadados(meta_tags, visiveis)
            rows.append(row_from_meta(merged, fallback=item))
        time.sleep(0.3)
    df = pd.DataFrame(rows)
    csv_path = os.path.join(OUTPUT_DIR, f"enriquecido_{datetime.now():%Y-%m-%d}.csv")
    df.to_csv(csv_path, index=False)
    log.info(f"CSV salvo em {csv_path}. Falhas: {len(erros)}")
    if erros:
        with open("erros_enriquecimento.log", "w") as f:
            f.write("\n".join(erros))
else:
    log.info("Enriquecimento desativado (MAX_ENRIQUECER = 0 ou None).")

2026-05-09 15:43:28,374 [INFO] Enriquecendo até 20 registros...
2026-05-09 15:43:28,375 [INFO] 1/9: As estratégias de recomposição e rearticulaçãodaselitesdo gr
2026-05-09 15:43:29,015 [INFO] 2/9: Espaço social e redes: contribuições metodológicas àsociolog
2026-05-09 15:43:30,299 [INFO] 3/9: As estratégias de recomposição e rearticulaçãodaselitesdo gr
2026-05-09 15:43:30,895 [INFO] 4/9: Sociologiadasreformas: orquestrando crítica,elitese especial
2026-05-09 15:43:32,337 [INFO] 5/9: Espaço social e redes: contribuições metodológicas àsociolog
2026-05-09 15:43:32,939 [INFO] 6/9: Apresentação: por um retorno àSociologiadasElites
2026-05-09 15:43:33,480 [INFO] 7/9: ELEMENTOS PARA UMASOCIOLOGIADASELITESARTÍSTICAS
2026-05-09 15:43:34,095 [INFO] 8/9: Sobre a construção do objeto emsociologia:: categorias bourd
2026-05-09 15:43:34,790 [INFO] 9/9: Monumentos do jornalismo brasileiro
2026-05-09 15:43:35,320 [INFO] CSV salvo em saida/enriquecido_2026-05-09.csv. Falhas: 0
